# Initialization
Turning the symlinks into proper paths the system can access. Finding the 16mixmux16 firmware.

In [1]:
from qick import QickSoc
import os
from pathlib import Path

base = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_16mixmux16")
out_dir = base / "out"

# 1. Clean up the non-working links
for name in ["qick_216.bit", "qick_216.hwh"]:
    f = out_dir / name
    if f.exists() or f.is_symlink():
        f.unlink()

# 2. Create actual, native OS-level relative symlinks
os.symlink("../top/top.runs/impl_1/d_1_wrapper.bit", str(out_dir / "qick_216.bit"))
os.symlink("../top/top.gen/sources_1/bd/d_1/hw_handoff/d_1.hwh", str(out_dir / "qick_216.hwh"))

print("Native Linux symlinks successfully recreated!")

# Load the firmware bitstream
soc = QickSoc("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_16mixmux16/out/qick_216.bit", external_clk=True)

soccfg = soc
print(soccfg)

out = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_16mixmux16/out")
for name in ["qick_216.bit", "qick_216.hwh"]:
    p = out / name
    if p.is_symlink():
        target = os.readlink(p)   # the link's path text, e.g. ../top/...
        p.unlink()                # removes only the link; the real file in top/ is untouched
        p.write_text(target)      # plain text placeholder, same as on the PC

Native Linux symlinks successfully recreated!
configuring reference clock chips, as requested
LMK04828 clock reference = 245.760 MHz, LMX2594 clock synth = 491.520 MHz
QICK running on ZCU216, software version 0.2.422

Firmware configuration (built Mon Jul 27 03:16:23 2026):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 2, DAC tile 3], [ADC tile 2]

	16 signal generator channels:
	0:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_sg_mixmux16_v1 - fs=6881.280 Msps, fabric=430.080 MHz
		32-bit DDS, range=430.080 MHz
		DAC tile 0, blk 2 is 2_228 on JHC1, or QICK box DAC port 2
	3:	axis_sg_mixmux16

In [2]:
# Jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *
from qick.asm_v2 import AveragerProgramV2 # still necessary even after importing all from QICK

# useful libraries
import numpy as np
import random 
import time
import math

# MixMuxed Signal Generation - Long Period For Spectrum Analysis

Similiar to code from Sho's Github, and copied from MixMux8SG.ipynb but modified. Implementing / testing the multi-tone generators. Readout is declared for the sake of running but is not used.

In [3]:
# --- Main MixMux16 program ---
class MixMuxProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=1, ro_ch=ro_ch, 
                         mixer_freq=cfg['mixer_freq'],
                         mux_freqs=cfg['pulse_freqs'], 
                         mux_gains=cfg['pulse_gains'], 
                         mux_phases=cfg['pulse_phases'])
        
        # Declaring unused dynamic readout
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['pulse_freqs'][0], phase=cfg['ro_phase'], gen_ch=gen_ch)

        self.add_pulse(ch=gen_ch, name="mymixmux", 
                       style="const", 
                       length=cfg["pulse_len"],
                       mask=cfg['pulse_mask'], # tones to play must be in the mask
                      )
        
    def _body(self, cfg):        
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True) # trigger readouts (unused)

        self.pulse(ch=cfg['gen_ch'], name="mymixmux", t=0) # trigger generator

In [4]:
gen_ch = 15 # which gen_ch to use

# --- Config dictionary ---
config = {'gen_ch': gen_ch,
          'ro_ch': 0, # unused readout channel
          'mixer_freq': 0,
          
          'pulse_freqs': [199.2, 199.3, 199.4, 199.5, 199.6, 199.7, 199.8, 199.9, 
                          200.0, 200.1, 200.2, 200.3, 200.4, 200.5, 200.6, 200.7], 
          'pulse_gains': [0.0625]*16,
          'pulse_phases': [0]*16,
          'pulse_mask': [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
          
          'ro_phase': 0.0,
          'trig_time': 0.4,
          'pulse_len': 4000000.0, # us
          'ro_len': 1.9,
         }

# --- Execution ---
prog = MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
iq_list = prog.acquire_decimated(soc, rounds=1)

# # --- Confirm completion and list frequencies ---
# print('Pulses should have successfully ran. Desired tones:')
# for i in config['pulse_freqs']:
#     print(i)

  0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
soc.reset_gens()

In [42]:
import copy

# Original config, unchanged except where the loop overrides it
orig_cfg = {
    'gen_ch': 4,
    'ro_ch': 0,
    'mixer_freq': 0,
    'pulse_freqs': [199.2 + 0.1 * i for i in range(16)],   # 199.2 ... 200.7 MHz
    'pulse_gains': [0.0625] * 16,                          # same gain per tone as the original
    'pulse_phases': [0] * 16,
    'pulse_mask': list(range(16)),
    'ro_phase': 0.0,
    'trig_time': 0.4,
    'pulse_len': 2.0e6,        # 2 s: long enough to check on a scope / analyzer
    'ro_len': 1.9,
}

results = {}   # n_tones -> True (saw output) / False (silent)

for n in range(1, 17):
    cfg = copy.deepcopy(orig_cfg)
    cfg['pulse_mask'] = list(range(n))   # enable only the first n lanes
    print(f"gen 4: {n} tone(s), lanes 0-{n-1}")
    try:
        MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=cfg).acquire_decimated(soc, rounds=1)
    except Exception as e:
        # A software error is a different failure from "ran but silent"
        print(f"  !! exception: {type(e).__name__}: {e}")
        results[n] = None
        continue
    # Record what you saw on the instrument
    ans = input("  Did you see output? (y/n): ").strip().lower()
    results[n] = (ans == 'y')

# Summary: which tone counts worked, and the first one that failed
print("\nSummary:", results)
failed = [n for n, ok in results.items() if ok is False]
print("First failing tone count:", failed[0] if failed else "none")

gen 4: 1 tone(s), lanes 0-0


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y
gen 4: 2 tone(s), lanes 0-1


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y
gen 4: 3 tone(s), lanes 0-2


...

  Did you see output? (y/n): n
gen 4: 15 tone(s), lanes 0-14


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): n
gen 4: 16 tone(s), lanes 0-15


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): n

Summary: {1: True, 2: True, 3: True, 4: True, 5: True, 6: True, 7: True, 8: True, 9: False, 10: False, 11: False, 12: False, 13: False, 14: False, 15: False, 16: False}
First failing tone count: 9


In [33]:
import copy

# orig_cfg is the same 16-tone config as before (gen 4, lane i fixed at 199.2 + 0.1*i MHz)
log = {}

def trial(label, lanes):
    """Play the given lanes on gen 4 and record whether output was seen."""
    cfg = copy.deepcopy(orig_cfg)
    cfg['gen_ch'] = 4
    cfg['pulse_mask'] = sorted(lanes)
    print(f"\n{label}: lanes {sorted(lanes)}")
    try:
        MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=cfg).acquire_decimated(soc, rounds=1)
    except Exception as e:
        print(f"  !! exception: {type(e).__name__}: {e}")
        log[label] = None
        return
    log[label] = input("  Did you see output? (y/n): ").strip().lower() == 'y'

# Phase 1: all 13 lanes that avoid 8, 9, 10. Prediction: plays.
safe = [l for l in range(16) if l not in (8, 9, 10)]
trial("safe 13 lanes", safe)

# Phase 2: add each suspect back to the safe set. Prediction: silent for the bad lane(s).
for bad in (8, 9, 10):
    trial(f"safe + {bad}", safe + [bad])

# Phase 3: add every other lane, one at a time, to a small base set.
# Shows which single additions break a set that otherwise works.
base = [0, 1, 2, 3]
trial("base only", base)
for k in range(4, 16):
    trial(f"base + {k}", base + [k])

print("\n--- Summary ---")
for label, ok in log.items():
    print(f"{label:16s} {'PLAYED' if ok else ('ERROR' if ok is None else 'SILENT')}")


safe 13 lanes: lanes [0, 1, 2, 3, 4, 5, 6, 7, 11, 12, 13, 14, 15]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

safe + 8: lanes [0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): n

safe + 9: lanes [0, 1, 2, 3, 4, 5, 6, 7, 9, 11, 12, 13, 14, 15]


...

  Did you see output? (y/n): y

base + 14: lanes [0, 1, 2, 3, 14]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

base + 15: lanes [0, 1, 2, 3, 15]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

--- Summary ---
safe 13 lanes    PLAYED
safe + 8         SILENT
safe + 9         SILENT
safe + 10        SILENT
base only        PLAYED
base + 4         PLAYED
base + 5         PLAYED
base + 6         PLAYED
base + 7         PLAYED
base + 8         SILENT
base + 9         SILENT
base + 10        SILENT
base + 11        PLAYED
base + 12        PLAYED
base + 13        PLAYED
base + 14        PLAYED
base + 15        PLAYED


In [36]:
import copy

def play(ch, lane_freqs, gain=0.0625, length_us=2.0e6):
    """Play {lane: freq_MHz} on one gen. Lane and frequency are set independently."""
    cfg = copy.deepcopy(orig_cfg)                  # orig_cfg from the earlier blocks
    cfg['gen_ch'] = ch
    freqs = list(cfg['pulse_freqs'])               # start from the default lane frequencies
    for lane, f in lane_freqs.items():
        freqs[lane] = f                            # override only the lanes under test
    cfg['pulse_freqs'] = freqs
    cfg['pulse_gains'] = [gain] * 16
    cfg['pulse_mask'] = sorted(lane_freqs)
    MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=cfg).acquire_decimated(soc, rounds=1)

log = {}
def trial(label, lane_freqs):
    print(f"\n{label}: {lane_freqs}")
    play(4, lane_freqs)
    log[label] = input("  Did you see output? (y/n): ").strip().lower() == 'y'

base = {0: 199.2, 1: 199.3, 2: 199.4, 3: 199.5}    # known-good base set

# A: a "bad" lane (8) carrying a "good" frequency.
#    Silent -> lane 8 itself is the problem. Plays -> it's the frequency.
trial("lane 8 @ 199.65", {**base, 8: 199.65})

# B: a "good" lane (4) carrying a "bad" frequency.
#    Silent -> the frequency is the problem. Plays -> it's the lane.
trial("lane 4 @ 200.1", {**base, 4: 200.1})

# C: sweep a known-good lane (4) across the suspect band and beyond.
for f in [199.6, 199.9, 200.0, 200.05, 200.1, 200.2, 200.3, 200.5, 201.0]:
    trial(f"lane 4 @ {f}", {**base, 4: f})

# D: the same sweep on a working gen (5) as a control.
print("\nControl on gen 5")
play(5, {**base, 4: 200.1})
log["gen 5 control"] = input("  Did you see output? (y/n): ").strip().lower() == 'y'

print("\n--- Summary ---")
for k, v in log.items():
    print(f"{k:22s} {'PLAYED' if v else 'SILENT'}")


lane 8 @ 199.65: {0: 199.2, 1: 199.3, 2: 199.4, 3: 199.5, 8: 199.65}


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

lane 4 @ 200.1: {0: 199.2, 1: 199.3, 2: 199.4, 3: 199.5, 4: 200.1}


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

lane 4 @ 199.6: {0: 199.2, 1: 199.3, 2: 199.4, 3: 199.5, 4: 199.6}


...

  Did you see output? (y/n): y

lane 4 @ 201.0: {0: 199.2, 1: 199.3, 2: 199.4, 3: 199.5, 4: 201.0}


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

Control on gen 5


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): n

--- Summary ---
lane 8 @ 199.65        PLAYED
lane 4 @ 200.1         PLAYED
lane 4 @ 199.6         PLAYED
lane 4 @ 199.9         PLAYED
lane 4 @ 200.0         PLAYED
lane 4 @ 200.05        PLAYED
lane 4 @ 200.2         PLAYED
lane 4 @ 200.3         PLAYED
lane 4 @ 200.5         PLAYED
lane 4 @ 201.0         PLAYED
gen 5 control          SILENT


In [44]:
import copy

# orig_cfg is the 16-tone config from before: lane i plays 199.2 + 0.1*i MHz.
# Frequencies stay fixed per lane, so only the lane selection changes.
GEN = 4
suspects = [8, 9, 10]                                    # lanes under test
outside = [l for l in range(16) if l not in suspects]    # the 13 lanes outside 8-10
PULSE_US = 1.0e6                                         # 1 s: long enough to check on a scope

results = {}   # tuple of lanes -> True (output seen) / False (silent) / None (exception)

def trial(lanes, label=""):
    """Play the given lanes on gen 4 and record whether output was seen."""
    lanes = tuple(sorted(lanes))
    cfg = copy.deepcopy(orig_cfg)
    cfg['gen_ch'] = GEN
    cfg['pulse_mask'] = list(lanes)      # only these lanes are enabled
    cfg['pulse_len'] = PULSE_US
    print(f"\n{label} lanes {list(lanes)}")
    try:
        MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=cfg).acquire_decimated(soc, rounds=1)
    except Exception as e:
        # A software error is a different failure from "ran but silent"
        print(f"  !! exception: {type(e).__name__}: {e}")
        results[lanes] = None
        return
    # Answer after watching the instrument for this run
    results[lanes] = input("  Did you see output? (y/n): ").strip().lower() == 'y'

# --- Controls: two outside-only lanes. These should play, otherwise the test setup is suspect ---
trial([0, 11], "CONTROL")
trial([4, 13], "CONTROL")

# --- Phase 1: each suspect lane on its own ---
for s in suspects:
    trial([s], "SOLO")

# --- Phase 2: each suspect lane + one outside lane (3 x 13 = 39 trials) ---
# To shorten the run, replace `outside` with a subset such as [0, 4, 7, 11, 15]
for s in suspects:
    for p in outside:
        trial([s, p], "PAIR")

# --- Summary ---
def mark(r):
    return " E" if r is None else (" P" if r else " S")

print("\n--- Controls and solo (P = played, S = silent, E = error) ---")
for key in [(0, 11), (4, 13)] + [(s,) for s in suspects]:
    print(f"  lanes {str(list(key)):10s} {mark(results[key])}")

print("\n--- Pair matrix: rows = suspect lane, columns = partner lane ---")
print("lane " + "".join(f"{p:>3d}" for p in outside))
for s in suspects:
    row = "".join(f"  {mark(results[tuple(sorted((s, p)))]).strip()}" for p in outside)
    print(f"{s:>4d} {row}")

# Partners that are silent with every suspect vs. only with some
for s in suspects:
    bad = [p for p in outside if results[tuple(sorted((s, p)))] is False]
    print(f"lane {s}: silent with partners {bad if bad else 'none'}")


CONTROL lanes [0, 11]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

CONTROL lanes [4, 13]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): y

SOLO lanes [8]


...

  Did you see output? (y/n): n

PAIR lanes [10, 14]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): n

PAIR lanes [10, 15]


  0%|          | 0/1 [00:00<?, ?it/s]

  Did you see output? (y/n): n

--- Controls and solo (P = played, S = silent, E = error) ---
  lanes [0, 11]     P
  lanes [4, 13]     P
  lanes [8]         S
  lanes [9]         S
  lanes [10]        S

--- Pair matrix: rows = suspect lane, columns = partner lane ---
lane   0  1  2  3  4  5  6  7 11 12 13 14 15
   8   S  S  S  S  S  S  S  S  S  S  S  S  S
   9   S  S  S  S  S  S  S  S  S  S  S  S  S
  10   S  S  S  S  S  S  S  S  S  S  S  S  S
lane 8: silent with partners [0, 1, 2, 3, 4, 5, 6, 7, 11, 12, 13, 14, 15]
lane 9: silent with partners [0, 1, 2, 3, 4, 5, 6, 7, 11, 12, 13, 14, 15]
lane 10: silent with partners [0, 1, 2, 3, 4, 5, 6, 7, 11, 12, 13, 14, 15]


In [45]:
import copy

def play(gen, lanes, length_us=1.0e6, gains=None):
    """Play the given lanes on one gen. orig_cfg is the 16-tone config (lane i = 199.2 + 0.1*i MHz)."""
    cfg = copy.deepcopy(orig_cfg)
    cfg['gen_ch'] = gen
    cfg['pulse_mask'] = sorted(lanes)
    cfg['pulse_len'] = length_us
    if gains is not None:
        cfg['pulse_gains'] = gains
    MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=cfg).acquire_decimated(soc, rounds=1)

# Same two sets, five times each, alternating. One should always play, one always fail.
for rep in range(5):
    for name, lanes in [("good [0-3]", [0, 1, 2, 3]), ("bad [0-3,8]", [0, 1, 2, 3, 8])]:
        print(f"rep {rep}: gen 4, {name}")
        play(4, lanes)
        print("   ->", input("   output? (y/n): "))

rep 0: gen 4, good [0-3]


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): y
   -> y
rep 0: gen 4, bad [0-3,8]


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): n
   -> n
rep 1: gen 4, good [0-3]


...

   output? (y/n): n
   -> n
rep 4: gen 4, good [0-3]


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): y
   -> y
rep 4: gen 4, bad [0-3,8]


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): n
   -> n


In [47]:
# A: lane 8 in the mask, but with ZERO gain. Nothing extra is added to the sum.
#    Silent -> the mask bit itself (decode/packing) breaks gen 4, not the signal path.
gains = [0.0625] * 16
gains[8] = 0.0
print("A: mask has 8, gain[8]=0"); play(4, [0, 1, 2, 3, 8], gains=gains)
print("   ->", input("   output? (y/n): "))

# B: lane 8 NOT in the mask, but its gain is nonzero. Should be identical to the good set.
gains = [0.0625] * 16
print("B: mask without 8, gain[8] nonzero"); play(4, [0, 1, 2, 3], gains=gains)
print("   ->", input("   output? (y/n): "))

# C: mask has lane 8 but with its frequency far from the others, still with low gain
freqs_cfg = copy.deepcopy(orig_cfg)   # (optional extra: change pulse_freqs[8] if A plays)

A: mask has 8, gain[8]=0


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): n
   -> n
B: mask without 8, gain[8] nonzero


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): y
   -> y


In [48]:
def asm_for(gen, lanes):
    cfg = copy.deepcopy(orig_cfg)
    cfg['gen_ch'] = gen
    cfg['pulse_mask'] = sorted(lanes)
    prog = MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=cfg)
    return str(prog)           # QICK v2 programs print their assembly; adjust if your version differs

import difflib
a = asm_for(4, [0, 1, 2, 3, 8]).splitlines()
b = asm_for(5, [0, 1, 2, 3, 8]).splitlines()
print("\n".join(difflib.unified_diff(a, b, "gen4", "gen5", lineterm="")))

--- gen4
+++ gen5
@@ -1,11 +1,11 @@
 macros:
 	WriteReg(dst='s_core_w1', src=0)
-	Delay(auto=True, gens=True, ros=True, t=1.0, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7ce78f40>}, t_regs={'t': 430}, tag=None)
+	Delay(auto=True, gens=True, ros=True, t=1.0, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7ced0100>}, t_regs={'t': 430}, tag=None)
 	OpenLoop(n=1, name='reps')
-	Trigger(ddr4=True, mr=False, outdict=defaultdict(<class 'int'>, {}), pins=[0], ros=[0], t=0.4, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7ce78fd0>, 'width': <qick.asm_v2.QickParam object at 0xffff7ce78ca0>}, t_regs={'t': 172, 'width': 10}, tag=None, trigset={0, 9, 10}, tts=[], width=0.023251488095238096)
-	Pulse(ch=4, name='mymixmux', t=0, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7ce78d90>}, t_regs={'t': 0}, tag=None)
-	Wait(auto=True, gens=False, no_warn=True, ros=True, t=0, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7ce78a00>}, t_regs={'t': 990}, tag=None)
-	De

In [59]:
import copy

def play(gen, lanes, gains, length_us=1.0e6):
    cfg = copy.deepcopy(orig_cfg)
    cfg['gen_ch'] = gen
    cfg['pulse_mask'] = sorted(lanes)
    cfg['pulse_gains'] = gains
    cfg['pulse_len'] = length_us
    MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=cfg).acquire_decimated(soc, rounds=1)

def zero(*lanes):
    g = [0.0625] * 16
    for l in lanes:
        g[l] = 0.0
    return g

tests = [
    ("gen4 mask+11, gain 0 (control, expect play)", 4, [0,1,2,3,11], zero(11)),
    ("gen4 mask+15, gain 0 (control, expect play)", 4, [0,1,2,3,15], zero(15)),
    ("gen4 mask+8,  gain 0 (known silent)",         4, [0,1,2,3,8],  zero(8)),
    ("gen4 mask+9,  gain 0",                        4, [0,1,2,3,9],  zero(9)),
    ("gen4 mask+10, gain 0",                        4, [0,1,2,3,10], zero(10)),
    ("gen5 mask+8,  gain 0 (control, expect play)", 5, [0,1,2,3,8],  zero(8)),
]
for name, gen, lanes, gains in tests:
    print(name)
    play(gen, lanes, gains)
    print("   ->", input("   output? (y/n): "))

gen4 mask+11, gain 0 (control, expect play)


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): y
   -> y
gen4 mask+15, gain 0 (control, expect play)


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): y
   -> y
gen4 mask+8,  gain 0 (known silent)


...

   output? (y/n): n
   -> n
gen4 mask+10, gain 0


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): n
   -> n
gen5 mask+8,  gain 0 (control, expect play)


  0%|          | 0/1 [00:00<?, ?it/s]

   output? (y/n): n
   -> n


In [62]:
# List every generator that has a tmux_ch entry (expect only gen 4 if the theory holds)
for i, g in enumerate(soccfg['gens']):
    if 'tmux_ch' in g:
        print(i, g['fullpath'], 'tmux_ch =', g['tmux_ch'])

# # Find which other blocks mention tmux in your config
# print([k for k in soccfg.keys()])

4 axis_sg_mixmux16_v1_4 tmux_ch = 0
